# Biology — Leaf image annotation and lesion-region segmentation
## FDP Day 5 — AI-Powered Scientific Research with Google Colab

**Objective:** Compare manually annotated lesion-area metadata and use HSV colour segmentation to highlight candidate regions. This is not a disease diagnosis.

**Matching dataset:** `01_Biology_Leaf_Disease_Image_Metadata.csv`

All bundled CSVs are **synthetic teaching datasets** with plausible ranges and units, not real experimental observations. Replace them with approved research data for actual studies. Avoid uploading identifiable patient or confidential data.

**Workflow:** Upload → inspect → analyse → visualise → interpret → export.


## 1. Setup and upload the matching CSV

In [ ]:
!pip -q install seaborn scikit-learn opencv-python-headless
import io, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from IPython.display import display
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid",context="notebook")
plt.rcParams["figure.figsize"]=(9,5)
uploaded=files.upload()
csv_name=next(f for f in uploaded if f.lower().endswith(".csv"))
df=pd.read_csv(io.BytesIO(uploaded[csv_name]))
print("Loaded",csv_name,"shape:",df.shape)
display(df.head()); display(df.describe(include="all").T)


## 2. Upload image input
**Suggested image:** `01_sample_leaf_lesions.jpg`. Supported formats: JPG, JPEG, PNG. Use original-resolution, focused images with consistent lighting/magnification. The supplied image is an illustrative teaching sample.

In [ ]:
from PIL import Image
image_upload=files.upload()
image_name=next(f for f in image_upload if f.lower().endswith((".jpg",".jpeg",".png")))
img=np.array(Image.open(io.BytesIO(image_upload[image_name])).convert("RGB"))
print("Image shape:",img.shape)
plt.figure(figsize=(10,6));plt.imshow(img);plt.axis("off");plt.title("Uploaded image");plt.show()


## 3. Image annotation and candidate lesion segmentation
Tune HSV thresholds to your image. Colour-selected regions are not pathogen identification.

In [ ]:
import cv2
hsv=cv2.cvtColor(img,cv2.COLOR_RGB2HSV)
mask=cv2.inRange(hsv,np.array([5,35,25]),np.array([35,255,230]))
mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,np.ones((5,5),np.uint8))
overlay=img.copy();overlay[mask>0]=(255,40,40)
fig,ax=plt.subplots(1,3,figsize=(15,5))
for a,im_,t in zip(ax,[img,mask,overlay],["Original","Candidate mask","Highlighted regions"]):a.imshow(im_,cmap="gray" if t=="Candidate mask" else None);a.set_title(t);a.axis("off")
plt.tight_layout();plt.show()
print("Colour-selected pixel percentage:",round(100*np.mean(mask>0),2))
sns.boxplot(data=df,x="condition",y="annotated_lesion_area_percent");plt.title("Annotated lesion area by group");plt.xticks(rotation=15);plt.show()
display(df.groupby("condition").annotated_lesion_area_percent.agg(["count","mean","median","std"]).round(2))


## Research interpretation checklist
- State the research question, data source, sample size and units.
- Explain two findings supported by the charts.
- Identify uncertainty, confounders, measurement limitations and possible bias.
- State what additional data or validation is required before publication.
- Do not claim causation from correlation or clinical/biological diagnosis from simple image segmentation.


In [ ]:
# Export a copy of the data with any new analysis columns
df.to_csv("processed_research_data.csv",index=False)
files.download("processed_research_data.csv")
